# Comparing ML Techniques for Emotion Recognition
**Cognitive Science Course Project — Techniques Comparison**

We compare three approaches to emotion recognition, each modeling human cognition differently:

| Approach | Cognitive Analogy |
|---|---|
| HOG + SVM | Rule-based theories — fixed, hand-crafted features like early cognitive models |
| Custom CNN | Visual cortex hierarchy — learns features bottom-up, like the brain |
| Transfer Learning (MobileNetV2) | Prior experience — pre-trained knowledge shaping new perception |

## 1. Setup

In [ ]:
import os
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
from sklearn.utils.class_weight import compute_class_weight
import tensorflow as tf
from tensorflow.keras import layers, models, applications
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

EMOTIONS = ['Angry', 'Disgust', 'Fear', 'Happy', 'Sad', 'Surprise', 'Neutral']
TRAIN_PATH = '/kaggle/input/competitions/challenges-in-representation-learning-facial-expression-recognition-challenge/train.csv'

print('TensorFlow:', tf.__version__)
print('GPU:', tf.config.list_physical_devices('GPU'))


## 2. Load & Prepare Data

In [ ]:
df = pd.read_csv(TRAIN_PATH)

X_raw = np.array([
    np.array(row.split(), dtype=np.float32).reshape(48, 48)
    for row in df['pixels']
])
y_raw = df['emotion'].values

X_train_raw, X_val_raw, y_train, y_val = train_test_split(
    X_raw, y_raw, test_size=0.2, random_state=42
)

# Normalized versions for CNN
X_train_cnn = X_train_raw.reshape(-1, 48, 48, 1) / 255.0
X_val_cnn   = X_val_raw.reshape(-1, 48, 48, 1) / 255.0

y_train_cat = tf.keras.utils.to_categorical(y_train, 7)
y_val_cat   = tf.keras.utils.to_categorical(y_val, 7)

print(f'Train: {len(X_train_raw)} | Val: {len(X_val_raw)}')

In [ ]:
# Class weights to handle imbalance (Disgust: 75 samples vs Happy: ~1800)
class_weights_arr = compute_class_weight('balanced', classes=np.unique(y_train), y=y_train)
class_weight_dict = dict(enumerate(class_weights_arr))
print('Class weights:', {EMOTIONS[k]: f'{v:.2f}' for k, v in class_weight_dict.items()})


---
## Approach 1: HOG + SVM

**Cognitive analogy:** Early cognitive models of face recognition assumed fixed, rule-based feature detectors — like Gabor filters tuned to edges and textures. HOG (Histogram of Oriented Gradients) captures exactly this: local edge directions across the face. SVM then draws a decision boundary in that feature space.

This mirrors the idea that emotion recognition might rely on detecting specific facial geometry (raised brows = surprise, downturned mouth = sad) — a theory challenged by modern deep learning.

In [ ]:
from skimage.feature import hog
from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

def extract_hog(images):
    features = []
    for img in images:
        feat = hog(img, orientations=9, pixels_per_cell=(8, 8),
                   cells_per_block=(2, 2), visualize=False)
        features.append(feat)
    return np.array(features)

print('Extracting HOG features...')
X_train_hog = extract_hog(X_train_raw)
X_val_hog   = extract_hog(X_val_raw)
print(f'HOG feature vector size: {X_train_hog.shape[1]}')

In [ ]:
print('Training SVM (this takes a few minutes)...')
svm_pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('svm', SVC(kernel='rbf', C=10, gamma='scale', probability=True))
])
svm_pipeline.fit(X_train_hog, y_train)

joblib.dump(svm_pipeline, 'hog_svm_model.pkl')
print('Saved: hog_svm_model.pkl')

y_pred_svm = svm_pipeline.predict(X_val_hog)
svm_acc = accuracy_score(y_val, y_pred_svm)
print(f'\nHOG + SVM Accuracy: {svm_acc:.4f}')
print(classification_report(y_val, y_pred_svm, target_names=EMOTIONS))


---
## Approach 2: Custom CNN

**Cognitive analogy:** The CNN's layered architecture mirrors the visual cortex hierarchy — V1 detects edges, V2 combines them into shapes, IT cortex recognizes complex objects like faces. Unlike HOG, the CNN learns *which* features matter from data, much like how the brain's visual system is shaped by experience.

In [ ]:
def build_cnn():
    inputs = layers.Input(shape=(48, 48, 1))
    x = layers.RandomFlip('horizontal')(inputs)
    x = layers.RandomRotation(0.1)(x)
    x = layers.Conv2D(32, (3, 3), activation='relu')(x)
    x = layers.BatchNormalization()(x)
    x = layers.MaxPooling2D(2, 2)(x)
    x = layers.Dropout(0.25)(x)
    x = layers.Conv2D(64, (3, 3), activation='relu')(x)
    x = layers.BatchNormalization()(x)
    x = layers.MaxPooling2D(2, 2)(x)
    x = layers.Dropout(0.25)(x)
    x = layers.Conv2D(128, (3, 3), activation='relu')(x)
    x = layers.BatchNormalization()(x)
    x = layers.MaxPooling2D(2, 2)(x)
    x = layers.Dropout(0.4)(x)
    x = layers.Flatten()(x)
    x = layers.Dense(256, activation='relu')(x)
    x = layers.Dropout(0.5)(x)
    outputs = layers.Dense(7, activation='softmax')(x)
    model = models.Model(inputs, outputs)
    model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
    return model

cnn_model = build_cnn()
cnn_model.summary()


In [ ]:
cnn_callbacks = [
    EarlyStopping(monitor='val_accuracy', patience=10, restore_best_weights=True, verbose=1),
    ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=5, min_lr=1e-6, verbose=1),
]

cnn_history = cnn_model.fit(
    X_train_cnn, y_train_cat,
    epochs=75, batch_size=64,
    validation_data=(X_val_cnn, y_val_cat),
    class_weight=class_weight_dict,
    callbacks=cnn_callbacks,
)

cnn_model.save('cnn_emotion_model.keras')
print('Saved: cnn_emotion_model.keras')

y_pred_cnn = cnn_model.predict(X_val_cnn).argmax(axis=1)
cnn_acc = accuracy_score(y_val, y_pred_cnn)
print(f'\nCNN Accuracy: {cnn_acc:.4f}')
print(classification_report(y_val, y_pred_cnn, target_names=EMOTIONS))


---
## Approach 3: Transfer Learning (MobileNetV2)

**Cognitive analogy:** Humans don't learn to recognize emotions from scratch — we come pre-wired with face-processing mechanisms (the fusiform face area activates at birth) and refine them through experience. MobileNetV2, pre-trained on 1.2 million ImageNet images, has already learned rich visual representations. We fine-tune only the final layers — analogous to applying prior visual experience to a new specific task.

In [ ]:
# Resize to 96×96 — smallest MobileNetV2-supported size.
# At 48×48 the backbone outputs a 2×2 feature map, destroying all spatial info.
X_train_rgb = tf.image.resize(
    np.repeat(X_train_raw[..., np.newaxis], 3, axis=-1), (96, 96)
).numpy() / 255.0
X_val_rgb = tf.image.resize(
    np.repeat(X_val_raw[..., np.newaxis], 3, axis=-1), (96, 96)
).numpy() / 255.0

base_model = applications.MobileNetV2(
    input_shape=(96, 96, 3),
    include_top=False,
    weights='imagenet'
)
base_model.trainable = False

tl_model = models.Sequential([
    base_model,
    layers.GlobalAveragePooling2D(),
    layers.Dense(128, activation='relu'),
    layers.Dropout(0.4),
    layers.Dense(7, activation='softmax'),
])

tl_model.compile(
    optimizer=tf.keras.optimizers.Adam(1e-3),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)
tl_model.summary()


In [ ]:
# Phase 1: head only, base frozen
print('=== Phase 1: Training head (base frozen) ===')
tl_model.fit(
    X_train_rgb, y_train_cat,
    epochs=15, batch_size=64,
    validation_data=(X_val_rgb, y_val_cat),
    class_weight=class_weight_dict,
    callbacks=[EarlyStopping(monitor='val_accuracy', patience=5, restore_best_weights=True, verbose=1)],
)

# Phase 2: unfreeze last 30 base layers, fine-tune with low LR
print('\n=== Phase 2: Fine-tuning (last 30 base layers unfrozen) ===')
base_model.trainable = True
for layer in base_model.layers[:-30]:
    layer.trainable = False

tl_model.compile(
    optimizer=tf.keras.optimizers.Adam(1e-5),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

tl_history = tl_model.fit(
    X_train_rgb, y_train_cat,
    epochs=30, batch_size=32,
    validation_data=(X_val_rgb, y_val_cat),
    class_weight=class_weight_dict,
    callbacks=[
        EarlyStopping(monitor='val_accuracy', patience=8, restore_best_weights=True, verbose=1),
        ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=4, min_lr=1e-7, verbose=1),
    ],
)

tl_model.save('tl_emotion_model.keras')
print('Saved: tl_emotion_model.keras')

y_pred_tl = tl_model.predict(X_val_rgb).argmax(axis=1)
tl_acc = accuracy_score(y_val, y_pred_tl)
print(f'\nTransfer Learning Accuracy: {tl_acc:.4f}')
print(classification_report(y_val, y_pred_tl, target_names=EMOTIONS))


---
## Final Comparison

In [ ]:
results = {
    'HOG + SVM':          {'acc': svm_acc, 'preds': y_pred_svm},
    'Custom CNN':         {'acc': cnn_acc, 'preds': y_pred_cnn},
    'Transfer Learning':  {'acc': tl_acc,  'preds': y_pred_tl},
}

# Accuracy bar chart
plt.figure(figsize=(8, 5))
names = list(results.keys())
accs  = [results[n]['acc'] for n in names]
bars  = plt.bar(names, accs, color=['#4c72b0', '#dd8452', '#55a868'])
for bar, acc in zip(bars, accs):
    plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
             f'{acc:.1%}', ha='center', fontsize=12, fontweight='bold')
plt.title('Accuracy Comparison Across ML Techniques')
plt.ylabel('Validation Accuracy')
plt.ylim(0, 0.85)
plt.tight_layout()
plt.show()

In [ ]:
# Side-by-side confusion matrices
fig, axes = plt.subplots(1, 3, figsize=(22, 6))
for ax, (name, res) in zip(axes, results.items()):
    cm = confusion_matrix(y_val, res['preds'])
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=EMOTIONS, yticklabels=EMOTIONS, ax=ax)
    ax.set_title(f'{name}\nAccuracy: {res["acc"]:.1%}')
    ax.set_ylabel('True')
    ax.set_xlabel('Predicted')
plt.tight_layout()
plt.show()

---
## Cognitive Science Takeaways

**Memory link:**
- Across all three models, **Happy** and **Surprise** consistently score highest. This aligns with the memory-emotion literature: emotionally salient stimuli (positive and arousing) are encoded more strongly in memory (amygdala modulation of hippocampal consolidation — McGaugh, 2000).
- **Fear** and **Disgust** score lowest — mirroring findings that humans also confuse these two, especially under time pressure (Ekman & Friesen, 1978).

**ML techniques link:**
- HOG + SVM represents the **classical, feature-engineering paradigm** — analogous to early symbolic AI theories of cognition.
- The CNN represents **connectionist/neural** approaches — bottom-up learning that parallels developmental theories of perceptual learning.
- Transfer learning represents **experience-dependent plasticity** — prior knowledge accelerating new task acquisition, consistent with schema theory in cognitive psychology.

**Key insight:** The accuracy gap between approaches reflects how much of emotion recognition is *general visual processing* (captured by transfer learning) vs. *emotion-specific features* (what the CNN learns from scratch).